# Let's go PRO!

Advanced RAG Techniques!

Let's start by digging into ingest:

1. No LangChain! Just native for maximum flexibility
2. Let's use an LLM to divide up chunks in a sensible way
3. Let's use the best chunk size and encoder from yesterday
4. Let's also have the LLM rewrite chunks in a way that's most useful ("document pre-processing")

In [1]:
from pathlib import Path
from openai import OpenAI
from dotenv import load_dotenv
from pydantic import BaseModel, Field
from chromadb import PersistentClient
from tqdm import tqdm
from litellm import completion
import os
import numpy as np
from sklearn.manifold import TSNE
import plotly.graph_objects as go
from supabase import create_client
import json

load_dotenv(override=True)

MODEL = "gpt-4.1-mini"

embedding_model = "text-embedding-3-large"
KNOWLEDGE_BASE_PATH = Path("../../knowledge-base")
AVERAGE_CHUNK_SIZE = 500

openai = OpenAI()

In [2]:
SUPABASE_KEY = os.getenv("SUPABASE_KEY")
SUPABASE_URL = os.getenv("SUPABASE_INSURLLM_RAG_URL")
supabase = create_client(SUPABASE_URL, SUPABASE_KEY)
collection_name = "semantic_chunked_documents"

In [3]:
# Inspired by LangChain's Document - let's have something similar

class Result(BaseModel):
    page_content: str
    metadata: dict

In [4]:
# A class to perfectly represent a chunk

class Chunk(BaseModel):
    headline: str = Field(description="A brief heading for this chunk, typically a few words, that is most likely to be surfaced in a query")
    summary: str = Field(description="A few sentences summarizing the content of this chunk to answer common questions")
    original_text: str = Field(description="The original text of this chunk from the provided document, exactly as is, not changed in any way")

    def as_result(self, document):
        metadata = {"source": document["source"], "type": document["type"]}
        return Result(page_content=self.headline + "\n\n" + self.summary + "\n\n" + self.original_text,metadata=metadata)


class Chunks(BaseModel):
    chunks: list[Chunk]

## Three steps:

1. Fetch documents from the knowledge base, like LangChain did
2. Call an LLM to turn documents into Chunks
3. Store the Chunks in Chroma

That's it!

### Let's start with Step 1

In [5]:
def fetch_documents():
    """A homemade version of the LangChain DirectoryLoader"""

    documents = []

    for folder in KNOWLEDGE_BASE_PATH.iterdir():
        doc_type = folder.name
        for file in folder.rglob("*.md"):
            with open(file, "r", encoding="utf-8") as f:
                documents.append({"type": doc_type, "source": file.as_posix(), "text": f.read()})

    print(f"Loaded {len(documents)} documents")
    return documents

In [6]:
documents = fetch_documents()

Loaded 76 documents


### Donezo! On to Step 2 - make the chunks

In [7]:
def make_prompt(document):
    how_many = (len(document["text"]) // AVERAGE_CHUNK_SIZE) + 1
    return f"""
You take a document and you split the document into overlapping chunks for a KnowledgeBase.

The document is from the shared drive of a company called Insurellm.
The document is of type: {document["type"]}
The document has been retrieved from: {document["source"]}

A chatbot will use these chunks to answer questions about the company.
You should divide up the document as you see fit, being sure that the entire document is returned in the chunks - don't leave anything out.
This document should probably be split into {how_many} chunks, but you can have more or less as appropriate.
There should be overlap between the chunks as appropriate; typically about 25% overlap or about 50 words, so you have the same text in multiple chunks for best retrieval results.

For each chunk, you should provide a headline, a summary, and the original text of the chunk.
Together your chunks should represent the entire document with overlap.

Here is the document:

{document["text"]}

Respond with the chunks.
"""

In [8]:
print(make_prompt(documents[0]))


You take a document and you split the document into overlapping chunks for a KnowledgeBase.

The document is from the shared drive of a company called Insurellm.
The document is of type: products
The document has been retrieved from: ../../knowledge-base/products/Rellm.md

A chatbot will use these chunks to answer questions about the company.
You should divide up the document as you see fit, being sure that the entire document is returned in the chunks - don't leave anything out.
This document should probably be split into 8 chunks, but you can have more or less as appropriate.
There should be overlap between the chunks as appropriate; typically about 25% overlap or about 50 words, so you have the same text in multiple chunks for best retrieval results.

For each chunk, you should provide a headline, a summary, and the original text of the chunk.
Together your chunks should represent the entire document with overlap.

Here is the document:

# Product Summary

# Rellm: AI-Powered Enter

In [9]:
def make_messages(document):
    return [
        {"role": "user", "content": make_prompt(document)},
    ]

In [10]:
make_messages(documents[0])

[{'role': 'user',
  'content': "\nYou take a document and you split the document into overlapping chunks for a KnowledgeBase.\n\nThe document is from the shared drive of a company called Insurellm.\nThe document is of type: products\nThe document has been retrieved from: ../../knowledge-base/products/Rellm.md\n\nA chatbot will use these chunks to answer questions about the company.\nYou should divide up the document as you see fit, being sure that the entire document is returned in the chunks - don't leave anything out.\nThis document should probably be split into 8 chunks, but you can have more or less as appropriate.\nThere should be overlap between the chunks as appropriate; typically about 25% overlap or about 50 words, so you have the same text in multiple chunks for best retrieval results.\n\nFor each chunk, you should provide a headline, a summary, and the original text of the chunk.\nTogether your chunks should represent the entire document with overlap.\n\nHere is the document

In [11]:
def process_document(document):
    messages = make_messages(document)
    response = completion(model=MODEL, messages=messages, response_format=Chunks)
    reply = response.choices[0].message.content
    doc_as_chunks = Chunks.model_validate_json(reply).chunks
    return [chunk.as_result(document) for chunk in doc_as_chunks]

In [13]:
process_document(documents[0])

[Result(page_content='Introduction to Rellm and Its Purpose\n\nRellm is an AI-powered enterprise reinsurance product by Insurellm, designed to transform reinsurance operations. It leverages artificial intelligence to improve risk management, decision-making, and operational efficiency, offering seamless integrations and robust analytics for proactive portfolio management.\n\n# Product Summary\n\n# Rellm: AI-Powered Enterprise Reinsurance Solution\n\n## Summary\n\nRellm is an innovative enterprise reinsurance product developed by Insurellm, designed to transform the way reinsurance companies operate. Harnessing the power of artificial intelligence, Rellm offers an advanced platform that redefines risk management, enhances decision-making processes, and optimizes operational efficiencies within the reinsurance industry. With seamless integrations and robust analytics, Rellm enables insurers to proactively manage their portfolios and respond to market dynamics with agility.', metadata={'s

In [14]:
def create_chunks(documents):
    chunks = []
    for doc in tqdm(documents):
        chunks.extend(process_document(doc))
    return chunks

In [15]:
chunks = create_chunks(documents)

100%|██████████| 76/76 [25:43<00:00, 20.32s/it]


In [17]:
print(len(chunks))

595


### Well that was easy! If a bit slow.

In the python module version, I sneakily use the multi-processing Pool to run this in parallel,
but if you get a Rate Limit Error you can turn this off in the code.

### Finally, Step 3 - save the embeddings

In [21]:
def create_embeddings(chunks, batch_size=50):
    # Clear existing rows
    supabase.table(collection_name).delete().neq("id", -1).execute()

    texts = [chunk.page_content for chunk in chunks]
    emb = openai.embeddings.create(model=embedding_model, input=texts).data
    vectors = [e.embedding for e in emb]

    rows = [
        {
            "id": i,
            "content": texts[i],
            "embedding": vectors[i],
            "metadata": chunks[i].metadata,
        }
        for i in range(len(chunks))
    ]

    # Insert in batches to avoid timeout
    for i in range(0, len(rows), batch_size):
        batch = rows[i:i + batch_size]
        supabase.table(collection_name).upsert(batch).execute()
        print(f"Inserted {min(i + batch_size, len(rows))}/{len(rows)} rows")

    print(f"Vectorstore created with {len(rows)} documents")

In [22]:
create_embeddings(chunks)

Inserted 50/595 rows
Inserted 100/595 rows
Inserted 150/595 rows
Inserted 200/595 rows
Inserted 250/595 rows
Inserted 300/595 rows
Inserted 350/595 rows
Inserted 400/595 rows
Inserted 450/595 rows
Inserted 500/595 rows
Inserted 550/595 rows
Inserted 595/595 rows
Vectorstore created with 595 documents


# Nothing more to do here... right?

Wait! Didja think I'd forget??

In [ ]:
def fetch_collection() -> tuple[list[str], list[str], np.ndarray]:

    result = supabase.table(collection_name).select("id, content, embedding, metadata").execute()

    documents  = [row["content"]   for row in result.data]
    metadatas  = [row["metadata"]  for row in result.data]
    # Parse string embeddings to float arrays
    vectors = np.array([
        json.loads(r["embedding"]) if isinstance(r["embedding"], str) else r["embedding"]
        for r in result.data
    ], dtype=np.float32)

    return documents, metadatas, vectors

In [ ]:

documents, metadatas, vectors = fetch_collection()
doc_types = [metadata['type'] for metadata in metadatas]
colors = [['blue', 'green', 'red', 'orange'][['products', 'employees', 'contracts', 'company'].index(t)] for t in doc_types]

In [ ]:
tsne = TSNE(n_components=2, random_state=42)
reduced_vectors = tsne.fit_transform(vectors)

# Create the 2D scatter plot
fig = go.Figure(data=[go.Scatter(
    x=reduced_vectors[:, 0],
    y=reduced_vectors[:, 1],
    mode='markers',
    marker=dict(size=5, color=colors, opacity=0.8),
    text=[f"Type: {t}<br>Text: {d[:100]}..." for t, d in zip(doc_types, documents)],
    hoverinfo='text'
)])

fig.update_layout(title='2D Supabase Vector Store Visualization',
    scene=dict(xaxis_title='x',yaxis_title='y'),
    width=800,
    height=600,
    margin=dict(r=20, b=10, l=10, t=40)
)

fig.show()

In [ ]:
tsne = TSNE(n_components=3, random_state=42)
reduced_vectors = tsne.fit_transform(vectors)

# Create the 3D scatter plot
fig = go.Figure(data=[go.Scatter3d(
    x=reduced_vectors[:, 0],
    y=reduced_vectors[:, 1],
    z=reduced_vectors[:, 2],
    mode='markers',
    marker=dict(size=5, color=colors, opacity=0.8),
    text=[f"Type: {t}<br>Text: {d[:100]}..." for t, d in zip(doc_types, documents)],
    hoverinfo='text'
)])

fig.update_layout(
    title='3D Supabase Vector Store Visualization',
    scene=dict(xaxis_title='x', yaxis_title='y', zaxis_title='z'),
    width=900,
    height=700,
    margin=dict(r=10, b=10, l=10, t=40)
)

fig.show()

## And now - let's build an Advanced RAG!

We will use these techniques:

1. Reranking - reorder the rank results
2. Query re-writing

In [ ]:
class RankOrder(BaseModel):
    order: list[int] = Field(
        description="The order of relevance of chunks, from most relevant to least relevant, by chunk id number"
    )

In [ ]:
def rerank(question, chunks):
    system_prompt = """
You are a document re-ranker.
You are provided with a question and a list of relevant chunks of text from a query of a knowledge base.
The chunks are provided in the order they were retrieved; this should be approximately ordered by relevance, but you may be able to improve on that.
You must rank order the provided chunks by relevance to the question, with the most relevant chunk first.
Reply only with the list of ranked chunk ids, nothing else. Include all the chunk ids you are provided with, reranked.
"""
    user_prompt = f"The user has asked the following question:\n\n{question}\n\nOrder all the chunks of text by relevance to the question, from most relevant to least relevant. Include all the chunk ids you are provided with, reranked.\n\n"
    user_prompt += "Here are the chunks:\n\n"
    for index, chunk in enumerate(chunks):
        user_prompt += f"# CHUNK ID: {index + 1}:\n\n{chunk.page_content}\n\n"
    user_prompt += "Reply only with the list of ranked chunk ids, nothing else."
    messages = [
        {"role": "system", "content": system_prompt},
        {"role": "user", "content": user_prompt},
    ]
    response = completion(model=MODEL, messages=messages, response_format=RankOrder)
    reply = response.choices[0].message.content
    order = RankOrder.model_validate_json(reply).order
    print(order)

    valid_ids = set(range(1, len(chunks) + 1))
    seen = set()
    safe_order = [i for i in order if i in valid_ids and not (i in seen or seen.add(i))]
    return [chunks[i - 1] for i in safe_order]

In [ ]:
RETRIEVAL_K = 20

def fetch_context_unranked(question):
    query_vec = openai.embeddings.create(model=embedding_model, input=[question]).data[0].embedding  #removed , dimensions=1536
    query_str = "[" + ",".join(str(x) for x in query_vec) + "]"

    results = supabase.rpc("match_semantic_chunked_documents", {
        "query_embedding": query_str,
        "query_text": question,   # websearch_to_tsquery handles natural language better
        "match_count": RETRIEVAL_K,
    }).execute()

    return [
        Result(page_content=row["content"], metadata=row["metadata"])
        for row in results.data
    ]

In [ ]:
question = "Who won the IIOTY award?"
chunks_for_question = fetch_context_unranked(question)

In [ ]:
for chunk in chunks_for_question:
    print(chunk.page_content[:100]+"...")

In [ ]:
reranked = rerank(question, chunks_for_question)

In [ ]:
for chunk in reranked:
    print(chunk.page_content[:100]+"...")

In [ ]:
question = "Who went to Manchester University?"
query_vec = openai.embeddings.create(model=embedding_model, input=[question]).data[0].embedding  #removed , dimensions=1536
query_str = "[" + ",".join(str(x) for x in query_vec) + "]"

results = supabase.rpc("match_semantic_chunked_documents", {
    "query_embedding": query_str,
    "query_text": question,
    "match_count": RETRIEVAL_K,
}).execute()

print(f"Total results: {len(results.data)}")
print(f"IDs returned: {[row['id'] for row in results.data]}")

In [ ]:
question = "Who went to Manchester University?"
RETRIEVAL_K = 20
chunks = fetch_context_unranked(question)

for index, c in enumerate(chunks):
    if "manchester" in c.page_content.lower():
        print(index)

In [ ]:
reranked = rerank(question, chunks)
for chunk in reranked:
    print(chunk.page_content[:50]+"...")

In [ ]:
for index, c in enumerate(reranked):
    if "manchester" in c.page_content.lower():
        print(index)

In [ ]:
reranked[0].page_content

In [ ]:
def fetch_context(question):
    chunks = fetch_context_unranked(question)
    return rerank(question, chunks)

In [ ]:
SYSTEM_PROMPT = """
You are a knowledgeable, friendly assistant representing the company Insurellm.
You are chatting with a user about Insurellm.
Your answer will be evaluated for accuracy, relevance and completeness, so make sure it only answers the question and fully answers it.
If you don't know the answer, say so.
For context, here are specific extracts from the Knowledge Base that might be directly relevant to the user's question:
{context}

With this context, please answer the user's question. Be accurate, relevant and complete.
"""

In [ ]:
# In the context, include the source of the chunk

def make_rag_messages(question, history, chunks):
    context = "\n\n".join(f"Extract from {chunk.metadata['source']}:\n{chunk.page_content}" for chunk in chunks)
    system_prompt = SYSTEM_PROMPT.format(context=context)
    return [{"role": "system", "content": system_prompt}] + history + [{"role": "user", "content": question}]

In [ ]:
def rewrite_query(question, history=[]):
    """Rewrite the user's question to be a more specific question that is more likely to surface relevant content in the Knowledge Base."""
    message = f"""
You are in a conversation with a user, answering questions about the company Insurellm.
You are about to look up information in a Knowledge Base to answer the user's question.

This is the history of your conversation so far with the user:
{history}

And this is the user's current question:
{question}

Respond only with a single, refined question that you will use to search the Knowledge Base.
It should be a VERY short specific question most likely to surface content. Focus on the question details.
Don't mention the company name unless it's a general question about the company.
IMPORTANT: Respond ONLY with the knowledgebase query, nothing else.
"""
    response = completion(model=MODEL, messages=[{"role": "system", "content": message}])
    return response.choices[0].message.content

In [ ]:
rewrite_query("Who won the IIOTY award?", [])

In [ ]:
def answer_question(question: str, history: list[dict] = []) -> tuple[str, list]:
    """
    Answer a question using RAG and return the answer and the retrieved context
    """
    query = rewrite_query(question, history)
    print(query)
    chunks = fetch_context(query)

    
    messages = make_rag_messages(question, history, chunks)
    response = completion(model=MODEL, messages=messages)
    return response.choices[0].message.content, chunks

In [ ]:
answer_question("Who won the IIOTY award?", [])

In [ ]:
rewrite_query("Who went to Manchester University?", [])

In [ ]:
answer_question("Who went to Manchester University?", [])

## Results with dimensions of 1536, using gpt-4.1-nano and K_RETRIEVAL = 20
Mean Reciprocal Rank (MRR)
0.8318
Normalized DCG (nDCG)
0.8146
Keyword Coverage
87.3%

## Results with dimensions of 1536, using gpt-4.1-mini and K_RETRIEVAL = 20 and turning OFF query re-writing
Mean Reciprocal Rank (MRR)
0.9151
Normalized DCG (nDCG)
0.8990
Keyword Coverage
95.6%

## Results with dimensions of 3072, using gpt-4.1-mini and K_RETRIEVAL = 20 and turning OFF query re-writing and removing the index because postgres only support vectors with upto 2000 dimensions
Mean Reciprocal Rank (MRR)
0.9345
Normalized DCG (nDCG)
0.9185
Keyword Coverage
97.9%

## Results with dimensions of 3072, using gpt-4.1-mini and K_RETRIEVAL = 20 and turning ON query re-writing and removing the index because postgres only support vectors with upto 2000 dimensions
Mean Reciprocal Rank (MRR)
0.9353
Normalized DCG (nDCG)
0.9183
Keyword Coverage
97.9%

## Results with dimensions of 3072, using gpt-4.1-mini and K_RETRIEVAL = 20 and turning ON query re-writing and removing the index because postgres only support vectors with upto 2000 dimensions. Also modified the supabase matching function with the following change
##          ts_rank(to_tsvector('english', s.content), q.q) +
###         case when s.content ilike '%' || query_text || '%' then 0.7 else 0 end as similarity
Mean Reciprocal Rank (MRR)
0.9402
Normalized DCG (nDCG)
0.9216
Keyword Coverage
98.7%


## Results with dimensions of 3072, using gpt-4.1-mini and K_RETRIEVAL = 40 and turning ON query re-writing 
### Also added the following line in the rewrite_query() prompt
###     Rewrite specifically to expand named entities ("IIOTY winner" → "Maxine Thompson IIOTY")
Mean Reciprocal Rank (MRR)
0.9408
Normalized DCG (nDCG)
0.9240
Keyword Coverage
98.7%

## ANSWER EVALUATION with the ^^^
Accuracy
4.83/5
Completeness
4.63/5
Relevance
4.61/5
